# LangGraph + LangChain
* 15.1에서 만든 **Node · Graph**와 15.2의 **`add_messages` · Conditional Edge**에 LangChain LLM을 연결합니다.

---
* `day15` Conda 환경을 사용합니다.
* LangChain OpenAI 연동 패키지를 추가로 설치합니다.

In [1]:
#!uv add langchain-openai

In [2]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)

WORKDIR : d:\autornd\SK Autonomous R&D\실습\15일차_실습


---
## Node 안에서 LangChain LLM 호출

Node 안에서 **`ChatOpenAI.invoke()`** 로 실제 LLM 응답을 받습니다.

| Message 타입 | 역할 |
|----------------|------|
| `HumanMessage` | 사용자 메시지 |
| `AIMessage` | 모델 응답 메시지 |
| `SystemMessage` | 시스템 프롬프트 (역할·규칙 지정) |

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0.7, api_key=OPENAI_API_KEY)

# 그래프 밖에서 먼저 단독 호출해 봅니다
demo_reply = llm.invoke([
    SystemMessage(content='한 문장으로만 답하세요.'),
    HumanMessage(content='LangGraph가 뭐야?'),
])
demo_reply.content

d:\autornd\SK Autonomous R&D\AutoRnDEnv_tutorial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'LangGraph는 LangChain 팀이 만든 그래프 기반(상태 머신/노드-엣지) 오케스트레이션 프레임워크로, 여러 단계의 LLM 실행 흐름을 분기·반복·상태관리까지 포함해 안정적으로 설계하고 제어할 수 있게 해줍니다.'

* Node 내부에서 LangChain 기반 LLM을 실행해 보겠습니다

In [4]:
from pydantic import BaseModel

class ChatState(BaseModel):
    question: str = ''
    answer: str = ''

In [5]:
def chat_node(state: ChatState) -> dict:
    """LangChain LLM을 Node 안에서 1회 호출합니다."""
    response = llm.invoke([
        SystemMessage(content='친절한 한국어 어시스턴트입니다. 2문장 이내로 답하세요.'),
        HumanMessage(content=state.question),
    ])
    return {'answer': response.content}

In [6]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ChatState)
workflow.add_node('chat', chat_node)
workflow.add_edge(START, 'chat')
workflow.add_edge('chat', END)

single_app = workflow.compile()

In [7]:
result = single_app.invoke(ChatState(question='LangGraph의 Node는 무엇을 하나요?'))
print('질문:', result['question'])
print('답변:', result['answer'])

질문: LangGraph의 Node는 무엇을 하나요?
답변: LangGraph의 **Node**는 그래프에서 실행되는 **단위 작업(함수/로직)**으로, 입력 상태를 받아 필요한 처리를 한 뒤 그 결과를 **다음 노드로 전달**합니다. 즉, 에이전트의 흐름(분기/루프 포함)을 만들기 위한 **상태 전이 단계**를 담당합니다.


In [8]:
for chunk in single_app.stream(ChatState(question='LangGraph에서 Edge는 무엇인가요?')):
    print(chunk)

{'chat': {'answer': 'LangGraph에서 **Edge**는 한 노드(작업/단계)에서 다른 노드로 **흐름이 이동하는 연결(전이 규칙)**을 의미합니다. 즉, 그래프가 다음에 어떤 노드를 실행할지 정하는 “화살표/전환” 설정입니다.'}}


---
## `chat_history`에 LangChain 메시지 저장

LangChain의 `HumanMessage` / `AIMessage`를 그대로 history에 쌓고,  
다음 턴에 **전체 history를 LLM에 넘겨** 맥락을 유지합니다.

In [9]:
from typing import Annotated

from pydantic import Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class ChatState(BaseModel):
    # 15.2와 동일: add_messages → Node는 새 메시지 리스트만 반환
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    question: str = ''  # 이번 턴에 추가할 사용자 질문

In [10]:
def multi_chat_node(state: ChatState) -> dict:
    # 이번 턴 사용자 메시지를 history에 추가
    user_msg = HumanMessage(content=state.question) # -> HumanMessage: 사용자(인간)의 메시지를 저장하는 메시지 클래스
    messages_for_llm = state.chat_history + [user_msg]

    # 지금까지의 대화 전체를 LLM에 전달
    response = llm.invoke([
        SystemMessage(content='친절한 한국어 어시스턴트입니다. 이전 대화를 기억하세요.'),
        *messages_for_llm,
    ])

    # add_messages 덕분에 [user_msg, response]만 반환해도 기존 history에 이어 붙음
    return {'chat_history': [user_msg, response]}


multi_workflow = StateGraph(ChatState)
multi_workflow.add_node('chat', multi_chat_node)
multi_workflow.add_edge(START, 'chat')
multi_workflow.add_edge('chat', END)

multi_app = multi_workflow.compile()

## 이름 기억 테스트

In [11]:
state = multi_app.invoke(ChatState(question='내 이름은 민수야. 기억해줘.'))

print('--- 1턴 ---')
for msg in state['chat_history']:
    role = '사용자' if msg.type == 'human' else 'AI'
    print(f'[{role}] {msg.content}')

--- 1턴 ---
[사용자] 내 이름은 민수야. 기억해줘.
[AI] 민수님, 알겠어요! 😊  
앞으로 대화할 때 **민수**님이라고 부를게요.


In [12]:
# 이전 state를 이어서 새 질문만 넣습니다
state = multi_app.invoke(ChatState(
    chat_history=state['chat_history'],
    question='내 이름이 뭐였지?',
))

print('--- 2턴 이후 전체 history ---')
for msg in state['chat_history']:
    role = '사용자' if msg.type == 'human' else 'AI'
    print(f'[{role}] {msg.content}')

--- 2턴 이후 전체 history ---
[사용자] 내 이름은 민수야. 기억해줘.
[AI] 민수님, 알겠어요! 😊  
앞으로 대화할 때 **민수**님이라고 부를게요.
[사용자] 내 이름이 뭐였지?
[AI] 민수님이에요! 😊


### `add_messages`가 필요한 이유

add_message 속성 추가 없이 Node가 **새 메시지만** 반환하면  
기존 history가 **통째로 덮어써져** 이전 대화가 사라집니다.

In [13]:
class BrokenChatState(BaseModel):
    chat_history: list[BaseMessage] = Field(default_factory=list) # -> Annotated[, add_messages] 없음
    question: str = ''


def broken_chat_node(state: BrokenChatState) -> dict:
    user_msg = HumanMessage(content=state.question)
    response = llm.invoke([HumanMessage(content=state.question)])
    return {'chat_history': [user_msg, response]}  # 기존 history 무시 → 덮어쓰기


broken_app = StateGraph(BrokenChatState)
broken_app.add_node('chat', broken_chat_node)
broken_app.add_edge(START, 'chat')
broken_app.add_edge('chat', END)
broken_app = broken_app.compile()

s1 = broken_app.invoke(BrokenChatState(question='내 이름은 영희야.'))
s2 = broken_app.invoke(BrokenChatState(
    chat_history=s1['chat_history'],
    question='내 이름이 뭐였지?',
))

print('history 메시지 수:', len(s2['chat_history']))
print('마지막 AI 답:', s2['chat_history'][-1].content)

history 메시지 수: 2
마지막 AI 답: 제가 지금 대화에서 당신 이름을 알 수 있는 정보는 없어요.  
이름을 알려주시면 그 뒤로 기억하고 부를게요.


### `input()` 기반으로 채팅하기
* LangGraph로 완전한 멀티턴 대화를 구현하려면, 대화 지속/종료를 선택하는 분기가 필요합니다
* `add_conditional_edges`와 적절한 `router`를 활용해 `input()` 기반으로 채팅을 구현해 보세요

In [14]:
from typing import Annotated

from pydantic import Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class ChatState(BaseModel):
    # 15.2와 동일: add_messages → Node는 새 메시지 리스트만 반환
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    question: str = ''  # 이번 턴에 추가할 사용자 질문
    end: bool = False   # True면 대화 종료


def user_node(state: ChatState) -> dict:
    """input()으로 사용자 입력을 받고, 종료 여부를 state에 기록합니다."""
    user_input = input('\n[User] (종료: q): ').strip()
    if user_input.lower() == 'q':
        print('대화를 종료합니다.')
        return {'question': user_input, 'end': True}
    return {'question': user_input, 'end': False}


def chat_node(state: ChatState) -> dict:
    # 이번 턴 사용자 메시지를 history에 추가
    user_msg = HumanMessage(content=state.question)
    messages_for_llm = state.chat_history + [user_msg]

    # 지금까지의 대화 전체를 LLM에 전달
    response = llm.invoke([
        SystemMessage(content='You are a kind AI Assistant. You MUST answer in Korean Language.'),
        *messages_for_llm,
    ])
    print(f'[AI]: {response.content}')

    # add_messages 덕분에 [user_msg, response]만 반환해도 기존 history에 이어 붙음
    return {'chat_history': [user_msg, response]}


multi_workflow = StateGraph(ChatState)
multi_workflow.add_node('user', user_node)
multi_workflow.add_node('chat', chat_node)
multi_workflow.add_edge(START, 'user')


In [15]:
def end_chat_router(state: ChatState) -> str:
    """종료 플래그가 True면 END, 아니면 chat 노드로 진행"""
    if state.end:
        return END
    return 'chat'


# user → (종료? END : chat), chat → user 로 순환
multi_workflow.add_conditional_edges('user', end_chat_router)
multi_workflow.add_edge('chat', 'user')

multi_app = multi_workflow.compile()


In [16]:
print('멀티턴 채팅 시작! 종료하려면 q를 입력하세요.')

# 그래프 내부에서 user ↔ chat 을 순환하므로 invoke 한 번으로 멀티턴 진행
final_state = multi_app.invoke(ChatState())

print('\n--- 대화 종료 ---')
print('총 메시지 수:', len(final_state['chat_history']))
for msg in final_state['chat_history']:
    role = 'User' if msg.type == 'human' else 'AI'
    print(f'[{role}] {msg.content}')


멀티턴 채팅 시작! 종료하려면 q를 입력하세요.
[AI]: 안녕하세요! 무엇을 도와드릴까요? 😊
[AI]: 좋아요! 😊  
원하시는 게 있으신가요?

예를 들면,
- 질문/상담
- 번역
- 글쓰기(자소서, 이메일, 보고서 등)
- 요약
- 공부 설명(수학/코딩/언어 등)
- 여행/일정 추천

하고 싶은 주제나 원하는 결과 형태를 알려주시면 바로 도와드릴게요.
[AI]: 알겠어요! 😊  
지금은 아직 요청 내용이 없는 것 같아요.

원하시는 걸 편하게 한 줄로만 적어주셔도 괜찮아요. 예를 들면:

1) “영어 이메일 하나 써줘”  
2) “이 문장 한국어로 번역해줘: …”  
3) “이 내용을 5줄로 요약해줘: …”  
4) “수학 문제 풀이해줘: …”  

어떤 도움 필요하실까요?
[AI]: 도와드릴 준비가 되어 있어요! 😊  
다만 아직 구체적인 요청이 없어서요.

원하시는 걸 아래 중 하나로 답해주셔도 좋아요(번호만 골라도 됩니다).

1) 번역  
2) 글쓰기(메일/자소서/보고서 등)  
3) 요약  
4) 문제 풀이/설명  
5) 아이디어/추천  
6) 기타(원하시는 내용 그대로 적어주세요)

어떤 걸 원하시나요?
[AI]: 좋아요! 😊  
지금은 요청이 비어 있어서, 제가 선택할 수 있게 딱 한 가지만 알려주세요.

**“어떤 걸 해줄까요?”**에 대해 아래처럼 한 줄로 답해주실 수 있을까요?

- 예: **“1번 번역. 이 문장 번역해줘: …”**  
- 예: **“2번 글쓰기. 회사에 보내는 이메일 써줘: …”**  
- 예: **“3번 요약. 아래 글 요약해줘: …”**

원하시는 항목(번역/글쓰기/요약/풀이/추천 중)과 함께 내용을 보내주시면 바로 시작할게요.
[AI]: 괜찮아요! 😊 아직 고르기 어렵다면, 아래 중 하나만 편하게 골라 답해 주세요.

**A) 번역**  
**B) 글쓰기(메일/자소서/보고서)**  
**C) 요약**  
**D) 문제 풀이/설명**  
**E) 추천/아이디어**

예: “C로 해줘(요약해줘)” 처럼요.  
어느

---
## 순환 그래프 — AI끼리 대화 시켜보기

15.2의 **A ↔ B ↔ judge** 순환 구조에 LangChain LLM을 넣습니다.

| Node | 역할 |
|------|------|
| `optimist` | 낙관론자 — 주제에 찬성·긍정 |
| `skeptic` | 회의론자 — 반박·비판 |

**중단 조건** (`debate_route`):
* `turn_count >= max_turns` → `END`
* 마지막 메시지에 `'합의'` 포함 → `END`
* 그 외 → `optimist`로 돌아가 순환

In [17]:
from typing import Annotated, Literal, Optional

from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0          # optimist가 말할 때마다 +1
    max_turns: int = 3          # optimist 3번 = 총 6메시지 입력 시 종료
    last_speaker: Literal['optimist', 'skeptic', 'moderator'] = 'skeptic'
    # 사회자 판정: 한쪽으로 기운 경우 True
    is_tilted: bool = False
    tilted_side: Optional[Literal['optimist', 'skeptic']] = None
    debate_summary: str = ''     # 최종 토론 요약


optimist_llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0.8, api_key=OPENAI_API_KEY)
skeptic_llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0.8, api_key=OPENAI_API_KEY)
moderator_llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0.8, api_key=OPENAI_API_KEY)


In [18]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            '당신은 낙관적인 토론자입니다. 주제에 찬성하는 입장을 말하고,'
            '상대 토론자의 주장이 있다면 반박하세요'
        )),
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제: {state.topic}')) # 첫 대화이면 토론 주제를 주고
    else:
        prompts.extend(state.chat_history) # 이전 대화가 있으면 이어서 대화

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = optimist_llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            '당신은 비판적인 토론자입니다. 상대 주장을 반박하세요. 2문장 이내. '
            '더 이상 반박이 어려워 패배를 인정한다면 "합의"라고 말하십시오'
        )),
        *state.chat_history,
    ]
    response = skeptic_llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

### `route` 함수와 Conditional Edge 구현
* `debate_route`는 **다음에 갈 Node 이름** 또는 `END`를 반환합니다.
* `add_conditional_edges('skeptic', debate_route)`

In [19]:
def debate_route(state: DebateState):
    if state.turn_count >= state.max_turns:
        return END
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '합의' in last_text:
        return END
    return 'optimist'


debate_workflow = StateGraph(DebateState)
debate_workflow.add_node('optimist', optimist_node)
debate_workflow.add_node('skeptic', skeptic_node)

debate_workflow.add_edge(START, 'optimist')
debate_workflow.add_edge('optimist', 'skeptic')
debate_workflow.add_conditional_edges('skeptic', debate_route)

debate_app = debate_workflow.compile()

In [20]:
init_debate = DebateState(topic='AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.', max_turns=10)

for chunk in debate_app.stream(init_debate):
    print(chunk)

{'optimist': {'chat_history': [AIMessage(content='저는 **“AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다”**에 찬성합니다. 그 이유는 AI가 단순히 기술의 진보가 아니라, **삶의 질을 높이는 방식으로 실제 문제를 해결**할 가능성이 크기 때문입니다.\n\n## 1) AI는 ‘불행의 원인’을 줄일 수 있다\n많은 불행은 질병, 빈곤, 돌봄 부족, 교육 격차, 불안정한 노동 같은 **구조적 문제**에서 옵니다. AI는 예를 들어:\n- 의료 분야에서 조기 진단·개인맞춤 치료를 돕고,\n- 행정·복지 서비스 접근성을 높이며,\n- 교육에서 학습 수준에 맞춘 맞춤형 학습을 제공하고,\n- 재난 예측이나 안전 관리로 피해를 줄이는 식으로  \n**사람들이 체감하는 고통과 스트레스의 빈도를 낮출 수** 있습니다.\n\n## 2) AI는 ‘시간과 선택권’을 늘려 행복에 직접 연결된다\n행복은 큰 성취만큼이나 **일상의 여유**에서 나오기도 합니다. AI가 반복 업무를 자동화하면, 사람들은 더 의미 있는 활동(가족 돌봄, 자기계발, 창의 활동)에 시간을 쓸 수 있습니다. 또한 개인에게 맞는 정보(식단, 운동, 학습, 생활 패턴)를 추천해 **의사결정 부담**도 줄일 수 있고요.\n\n## 3) 도구가 발전하면 ‘기회의 문’이 넓어진다\nAI가 잘 설계되면, 특정 능력이나 자원을 가진 사람만 누리던 서비스가 더 넓게 퍼집니다. 예를 들어:\n- 장애나 언어 장벽이 있는 사람도 AI 보조 도구로 더 쉽게 소통하고,\n- 초보자도 전문가 수준에 가까운 도움을 받아 역량을 키울 수 있습니다.  \n이건 “공정한 접근”을 강화하는 방향이라 행복에 긍정적입니다.\n\n---\n\n# 상대의 주장에 대한 반박(예상)\n상대가 “AI 발전은 일자리 감소·감시 강화·격차 확대를 낳아 행복을 해칠 것”이라고 말할 수 있습니다. 저는 여기서 다음처럼 반박하겠습니다.\n\n1) **일자리 변화는 ‘감소’가 아니라 ‘전환’의 문제일 수 있다** 

## 사회자 추가하기

토론 그래프에 **`moderator` Node**를 추가합니다.

* 매 라운드 끝에 양쪽 주장을 종합 (`moderator`)
* 한쪽으로 기운 경우 / 최대 턴 / 합의 → `summarize`에서 최종 요약 후 `END`
* 아직 팽팽하면 `optimist`로 돌아가 순환

```
START → optimist → skeptic → moderator ─┬→ optimist (계속)
                                        └→ summarize → END (종료)
```


In [21]:
from langchain_core.messages import AIMessage


class ModeratorVerdict(BaseModel):
    """사회자의 구조화 판정 결과"""
    round_summary: str = Field(description='이번 라운드 양쪽 주장 종합. 2문장 이내.')
    is_tilted: bool = Field(description='한쪽 주장이 명백히 우세해 토론이 기운 경우 True')
    tilted_side: Optional[Literal['optimist', 'skeptic']] = Field(
        default=None,
        description='기운 쪽. is_tilted가 True일 때만 설정',
    )


moderator_structured = moderator_llm.with_structured_output(ModeratorVerdict)


def moderator_node(state: DebateState) -> dict:
    verdict: ModeratorVerdict = moderator_structured.invoke([
        SystemMessage(content=(
            '당신은 공정한 사회자입니다.\n'
            '1) 지금까지의 양쪽 주장을 2문장 이내로 종합하세요.\n'
            '2) 한쪽이 논리·근거에서 명백히 우세해 토론이 한쪽으로 기운 경우에만 '
            'is_tilted=True로 두고 tilted_side를 지정하세요.\n'
            '3) 아직 팽팽하면 is_tilted=False로 두고, round_summary 끝에 '
            '각 토론자에게 던질 짧은 질문을 남기세요.'
        )),
        *state.chat_history,
    ])

    # 채팅 히스토리에는 사람이 읽기 쉬운 요약 메시지로 남김
    content = verdict.round_summary
    if verdict.is_tilted:
        side = '낙관론자' if verdict.tilted_side == 'optimist' else '회의론자'
        content += f' [판정: {side} 쪽으로 기운 것으로 판단 → 종료]'

    response = AIMessage(content=content, name='moderator')
    return {
        'chat_history': [response],
        'last_speaker': 'moderator',
        'is_tilted': verdict.is_tilted,
        'tilted_side': verdict.tilted_side,
    }


def summarize_node(state: DebateState) -> dict:
    """토론 종료 시 전체 대화를 최종 요약"""
    if state.is_tilted:
        side = '낙관론자(optimist)' if state.tilted_side == 'optimist' else '회의론자(skeptic)'
        reason = f'종료 사유: 토론이 {side} 쪽으로 기울었습니다.\n'
    elif state.turn_count >= state.max_turns:
        reason = '종료 사유: 최대 턴 수에 도달했습니다.\n'
    else:
        reason = '종료 사유: 합의에 도달했습니다.\n'

    summary = moderator_llm.invoke([
        SystemMessage(content=(
            '당신은 공정한 사회자입니다. 전체 토론을 3~5문장으로 요약하고, '
            '양쪽의 핵심 주장과 최종적으로 어느 쪽이 더 설득력 있었는지 정리하세요.'
        )),
        HumanMessage(content=f'토론 주제: {state.topic}\n{reason}'),
        *state.chat_history,
    ])
    summary.name = 'moderator'
    return {
        'chat_history': [summary],
        'debate_summary': summary.content,
        'last_speaker': 'moderator',
    }


def moderator_route(state: DebateState):
    """사회자 이후: 기운 경우/턴 초과/합의 → 요약 후 종료, 아니면 낙관론자로 순환"""
    if state.is_tilted:
        return 'summarize'
    if state.turn_count >= state.max_turns:
        return 'summarize'
    # skeptic이 합의를 말한 직후 moderator를 거친 경우도 종료
    skeptic_msgs = [m for m in state.chat_history if getattr(m, 'name', None) == 'skeptic']
    if skeptic_msgs and '합의' in skeptic_msgs[-1].content:
        return 'summarize'
    return 'optimist'


debate_workflow_rev = StateGraph(DebateState)
debate_workflow_rev.add_node('optimist', optimist_node)
debate_workflow_rev.add_node('skeptic', skeptic_node)
debate_workflow_rev.add_node('moderator', moderator_node)
debate_workflow_rev.add_node('summarize', summarize_node)

debate_workflow_rev.add_edge(START, 'optimist')
debate_workflow_rev.add_edge('optimist', 'skeptic')
debate_workflow_rev.add_edge('skeptic', 'moderator')  # 매 라운드 끝 → 사회자
debate_workflow_rev.add_conditional_edges('moderator', moderator_route)
debate_workflow_rev.add_edge('summarize', END)

debate_app_rev = debate_workflow_rev.compile()


In [22]:
init_debate = DebateState(topic='AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.', max_turns=10)

for chunk in debate_app_rev.stream(init_debate):
    print(chunk)

{'optimist': {'chat_history': [AIMessage(content='저는 **AI 발전은 인간의 행복에 긍정적인 영향을 줄 것**이라고 봅니다. 다만 “행복”이 단순히 편리함만이 아니라 **삶의 질, 안정감, 선택의 폭, 스트레스 감소** 같은 요소를 포함한다는 점에서, AI는 이를 전반적으로 개선할 가능성이 큽니다.\n\n## 1) 의료·복지에서 행복을 직접 끌어올림\nAI는 질병 조기진단, 개인 맞춤 치료, 의료 자원 배분 최적화에 크게 기여할 수 있습니다.  \n질병을 늦게 발견해 고통이 커지는 상황을 줄이고, 치료 효과를 높이면 그 자체로 행복(삶의 안정감, 고통 감소)에 긍정적입니다. “의료 혁신=행복 향상”은 설득력이 큰 연결고리입니다.\n\n## 2) 노동을 ‘대체’보다 ‘보완’하는 방향으로 가면 스트레스가 줄어듭니다\n상대는 AI가 일자리를 빼앗아 불행을 키운다고 주장할 수 있는데, 저는 그렇게 단선적으로 보지 않습니다.  \nAI는 반복 업무를 줄이고 사람은 더 창의적/대인적 업무에 집중하도록 돕는 **업무 재설계**가 가능하고, 전환 과정에서 **재교육·직무 전환**이 함께 이루어지면 오히려 실업 불안과 과로가 줄어들 수 있습니다. 행복은 “일을 하냐/못 하냐”뿐 아니라 **불안이 줄고 삶이 예측 가능해지는지**에도 달려 있는데, AI가 경제적·행정적 효율을 높이면 생활 안정성도 커집니다.\n\n## 3) 교육과 정보 접근성 확대는 ‘역량의 행복’을 만듭니다\nAI 튜터, 맞춤형 학습, 번역·장벽 완화는 학습 격차를 줄일 수 있습니다. 개인이 더 잘 이해하고 더 잘 해낼 수 있다는 감각(자기효능감)은 행복의 중요한 원천입니다.  \n“정보의 비대칭이 줄어드는 것”은 사회 전반의 좌절감을 낮추고 기회를 늘려 주기도 합니다.\n\n## 4) 단, 위험이 있다면 “부정론”이 아니라 “통제·설계”로 대응해야 합니다\n상대가 “AI가 프라이버시 침해, 차별, 감시 강화로 불행을 만든다”고 말할 수 있습니다. 이 지적은 일부 타당합